# Phase 4: Layer-wise Probing — Where Each Emotion Crystallizes

We train **linear probes** (logistic regression) at each of BERT's 13 hidden states
(embedding layer + 12 encoder layers) for each of the 28 emotions independently.

This reveals the **formation trajectory** of each emotion through the network:
at which layer does the representation become linearly separable for each emotion?

**Key questions:**
- Do all emotions crystallize at the same depth, or do some need more layers?
- Do frequent emotions crystallize earlier than rare ones?
- Does the crystallization layer predict vulnerability to compression?

In [ ]:
import sys, os, warnings, time
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.utils import compute_metrics

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 11})

device = 'cuda' if torch.cuda.is_available() else 'cpu'
results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

In [ ]:
# Load model and data
model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-final')
model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=NUM_LABELS, problem_type='multi_label_classification',
)
model.eval()
model.to(device)
print(f'Model loaded from {model_path}')

train_ds, val_ds, test_ds, _, data_collator = load_goemotions()
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

## 1. Extract Hidden States from All Layers

BERT has 13 hidden states: the embedding output + 12 encoder layer outputs.
We extract the **[CLS] token** representation from each layer for all examples.

In [ ]:
@torch.no_grad()
def extract_all_hidden_states(model, dataset, data_collator, batch_size=64):
    """Extract CLS hidden states from all 13 layers for all examples.
    
    Returns:
        hidden_states: list of 13 numpy arrays, each (N, 768)
        labels: numpy array (N, 28)
    """
    model.eval()
    dev = next(model.parameters()).device
    
    all_hidden = [[] for _ in range(13)]
    all_labels = []
    
    n_batches = (len(dataset) + batch_size - 1) // batch_size
    t0 = time.time()
    
    for i in range(0, len(dataset), batch_size):
        batch = data_collator([dataset[j] for j in range(i, min(i + batch_size, len(dataset)))])
        outputs = model(
            input_ids=batch['input_ids'].to(dev),
            attention_mask=batch['attention_mask'].to(dev),
            output_hidden_states=True,
        )
        # outputs.hidden_states: tuple of 13 tensors (batch, seq, 768)
        for layer_idx in range(13):
            cls = outputs.hidden_states[layer_idx][:, 0, :].cpu().numpy()
            all_hidden[layer_idx].append(cls)
        all_labels.append(batch['labels'].numpy())
        
        if (i // batch_size + 1) % 50 == 0:
            elapsed = time.time() - t0
            done = i // batch_size + 1
            print(f'  Batch {done}/{n_batches} ({elapsed:.0f}s)')
    
    hidden_states = [np.concatenate(h, axis=0) for h in all_hidden]
    labels = np.concatenate(all_labels, axis=0)
    print(f'  Done: {hidden_states[0].shape[0]} examples, {time.time()-t0:.0f}s')
    return hidden_states, labels

In [ ]:
print('Extracting hidden states from TRAIN set...')
hidden_train, labels_train = extract_all_hidden_states(model, train_ds, data_collator)
print(f'  Shape per layer: {hidden_train[0].shape}')

print('\nExtracting hidden states from TEST set...')
hidden_test, labels_test = extract_all_hidden_states(model, test_ds, data_collator)
print(f'  Shape per layer: {hidden_test[0].shape}')

# Free GPU memory
model.cpu()
if device == 'cuda':
    torch.cuda.empty_cache()
print('\nModel moved to CPU, GPU memory freed.')

## 2. Train Probing Classifiers

For each of 13 layers x 28 emotions = **364 probes**, we train a logistic regression
on the [CLS] embeddings. The probe F1 measures how linearly separable each emotion
is at each depth of the network.

In [ ]:
def train_all_probes(hidden_train, labels_train, hidden_test, labels_test):
    """Train linear probes: 13 layers x 28 emotions = 364 classifiers."""
    n_layers = len(hidden_train)
    n_emotions = labels_train.shape[1]
    probe_f1 = np.zeros((n_emotions, n_layers))
    
    t0 = time.time()
    for layer_idx in range(n_layers):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(hidden_train[layer_idx])
        X_test = scaler.transform(hidden_test[layer_idx])
        
        for emo_idx in range(n_emotions):
            y_train = labels_train[:, emo_idx].astype(int)
            y_test = labels_test[:, emo_idx].astype(int)
            
            if y_train.sum() < 5 or y_test.sum() < 5:
                probe_f1[emo_idx, layer_idx] = 0.0
                continue
            
            clf = LogisticRegression(
                max_iter=500, random_state=42, C=1.0, solver='lbfgs',
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            probe_f1[emo_idx, layer_idx] = f1_score(y_test, y_pred, zero_division=0)
        
        layer_name = 'Emb' if layer_idx == 0 else f'L{layer_idx - 1}'
        elapsed = time.time() - t0
        print(f'  {layer_name:>4s}: mean F1 = {probe_f1[:, layer_idx].mean():.4f}  ({elapsed:.0f}s)')
    
    return probe_f1

print('Training 364 probing classifiers...')
print('=' * 50)
probe_f1 = train_all_probes(hidden_train, labels_train, hidden_test, labels_test)
print(f'\nProbe matrix shape: {probe_f1.shape}  (emotions x layers)')

## 3. The Crystallization Map

Heatmap showing probe F1 for each emotion at each layer.
Emotions are sorted by their **crystallization layer** — the first layer where
the probe F1 reaches 80% of its maximum.

In [ ]:
# Compute crystallization layer for each emotion
def compute_crystallization_layer(probe_f1, threshold=0.8):
    """First layer where probe F1 >= threshold * max(F1 for that emotion)."""
    n_emotions, n_layers = probe_f1.shape
    crystal_layer = np.full(n_emotions, n_layers - 1)  # default: last layer
    
    for emo_idx in range(n_emotions):
        max_f1 = probe_f1[emo_idx].max()
        if max_f1 < 0.01:
            crystal_layer[emo_idx] = n_layers - 1
            continue
        target = threshold * max_f1
        for layer_idx in range(n_layers):
            if probe_f1[emo_idx, layer_idx] >= target:
                crystal_layer[emo_idx] = layer_idx
                break
    return crystal_layer

crystal_layer = compute_crystallization_layer(probe_f1)
layer_labels = ['Emb'] + [f'L{i}' for i in range(12)]

# Sort emotions by crystallization layer, then by max F1
sort_idx = np.lexsort((probe_f1.max(axis=1), crystal_layer))
sorted_emotions = [EMOTION_NAMES[i] for i in sort_idx]
sorted_f1 = probe_f1[sort_idx]

# Heatmap
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(
    sorted_f1, annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=layer_labels, yticklabels=sorted_emotions,
    vmin=0, vmax=sorted_f1.max() + 0.05,
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Probe F1', 'shrink': 0.6},
)

# Mark crystallization layer for each emotion
for i, orig_idx in enumerate(sort_idx):
    cl = crystal_layer[orig_idx]
    ax.add_patch(plt.Rectangle((cl, i), 1, 1, fill=False,
                                edgecolor='black', linewidth=2.5))

ax.set_xlabel('Layer', fontsize=13)
ax.set_title('Where Each Emotion Crystallizes in BERT\n'
             '(black box = crystallization layer: first layer at 80% of max F1)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'probing_crystallization_map.png'), bbox_inches='tight')
plt.show()
print('Saved: probing_crystallization_map.png')

## 4. Formation Trajectories

Line plots showing how probe F1 evolves across layers for selected emotions.

In [ ]:
# Select interesting emotions: mix of early/late crystallizers + high/low F1
max_f1_per_emo = probe_f1.max(axis=1)

# Pick: 2 earliest crystallizers, 2 latest, 2 highest F1, 2 lowest (with F1 > 0)
valid = max_f1_per_emo > 0.01
valid_indices = np.where(valid)[0]

earliest = valid_indices[np.argsort(crystal_layer[valid_indices])[:2]]
latest = valid_indices[np.argsort(crystal_layer[valid_indices])[-2:]]
highest = valid_indices[np.argsort(max_f1_per_emo[valid_indices])[-2:]]
lowest = valid_indices[np.argsort(max_f1_per_emo[valid_indices])[:2]]

selected = list(set(np.concatenate([earliest, latest, highest, lowest])))
selected.sort(key=lambda i: crystal_layer[i])

fig, ax = plt.subplots(figsize=(14, 7))
cmap = plt.cm.tab10

for i, emo_idx in enumerate(selected):
    ax.plot(range(13), probe_f1[emo_idx], 'o-', color=cmap(i % 10),
            label=f'{EMOTION_NAMES[emo_idx]} (crystal: {layer_labels[crystal_layer[emo_idx]]})',
            linewidth=2.5, markersize=6)

# Shade early/middle/late regions
ax.axvspan(0, 4.5, alpha=0.05, color='blue', label='_')
ax.axvspan(4.5, 8.5, alpha=0.05, color='green', label='_')
ax.axvspan(8.5, 12.5, alpha=0.05, color='red', label='_')
ax.text(2, ax.get_ylim()[1] * 0.95, 'Early', ha='center', fontsize=10, color='blue', alpha=0.5)
ax.text(6.5, ax.get_ylim()[1] * 0.95, 'Middle', ha='center', fontsize=10, color='green', alpha=0.5)
ax.text(10.5, ax.get_ylim()[1] * 0.95, 'Late', ha='center', fontsize=10, color='red', alpha=0.5)

ax.set_xticks(range(13))
ax.set_xticklabels(layer_labels)
ax.set_xlabel('Layer', fontsize=13)
ax.set_ylabel('Probe F1', fontsize=13)
ax.set_title('Emotion Formation Trajectories Through BERT', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, loc='center left', bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'probing_trajectories.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart: crystallization layer per emotion
valid_mask = max_f1_per_emo > 0.01
valid_emos = [(EMOTION_NAMES[i], crystal_layer[i], max_f1_per_emo[i])
              for i in range(NUM_LABELS) if valid_mask[i]]
valid_emos.sort(key=lambda x: (x[1], -x[2]))

fig, ax = plt.subplots(figsize=(12, 8))
colors = []
for _, cl, _ in valid_emos:
    if cl <= 4:
        colors.append('#2ecc71')  # early
    elif cl <= 8:
        colors.append('#f39c12')  # middle
    else:
        colors.append('#e74c3c')  # late

y_pos = range(len(valid_emos))
ax.barh(y_pos, [cl for _, cl, _ in valid_emos], color=colors, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels([e for e, _, _ in valid_emos], fontsize=10)
ax.set_xlabel('Crystallization Layer', fontsize=13)
ax.set_title('When Each Emotion Becomes Linearly Separable', fontsize=14, fontweight='bold')
ax.set_xticks(range(13))
ax.set_xticklabels(layer_labels)
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Early (Emb-L3)'),
    Patch(facecolor='#f39c12', label='Middle (L4-L7)'),
    Patch(facecolor='#e74c3c', label='Late (L8-L11)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'probing_crystallization_bars.png'), bbox_inches='tight')
plt.show()

## 5. What Determines Crystallization Depth?

Do frequent emotions crystallize earlier? Is there a relationship between
crystallization depth and the final probe quality?

In [ ]:
from scipy.stats import spearmanr, pearsonr

# Compute emotion frequency in training data
train_labels = np.array(labels_train)
emotion_freq = train_labels.sum(axis=0)  # count per emotion

valid_idx = np.where(valid_mask)[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: Crystallization layer vs frequency
ax = axes[0]
ax.scatter(emotion_freq[valid_idx], crystal_layer[valid_idx],
           c=max_f1_per_emo[valid_idx], cmap='RdYlGn', s=100,
           edgecolors='white', linewidth=1.5, zorder=3)
for i in valid_idx:
    ax.annotate(EMOTION_NAMES[i], (emotion_freq[i], crystal_layer[i]),
                fontsize=7, textcoords='offset points', xytext=(5, 3))
ax.set_xlabel('Training frequency (# examples)', fontsize=12)
ax.set_ylabel('Crystallization layer', fontsize=12)
ax.set_title('Does frequency predict depth?', fontsize=13, fontweight='bold')

rho, p = spearmanr(emotion_freq[valid_idx], crystal_layer[valid_idx])
ax.text(0.05, 0.95, f'Spearman r = {rho:.3f}\np = {p:.4f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Panel 2: Crystallization layer vs max probe F1
ax = axes[1]
ax.scatter(crystal_layer[valid_idx], max_f1_per_emo[valid_idx],
           c=emotion_freq[valid_idx], cmap='viridis', s=100,
           edgecolors='white', linewidth=1.5, zorder=3)
for i in valid_idx:
    ax.annotate(EMOTION_NAMES[i], (crystal_layer[i], max_f1_per_emo[i]),
                fontsize=7, textcoords='offset points', xytext=(5, 3))
ax.set_xlabel('Crystallization layer', fontsize=12)
ax.set_ylabel('Max probe F1', fontsize=12)
ax.set_title('Does depth predict probe quality?', fontsize=13, fontweight='bold')

rho2, p2 = spearmanr(crystal_layer[valid_idx], max_f1_per_emo[valid_idx])
ax.text(0.05, 0.95, f'Spearman r = {rho2:.3f}\np = {p2:.4f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'probing_correlations.png'), bbox_inches='tight')
plt.show()

## 6. Layer Information Gain

For each layer, how much **new information** does it add?
Measured as the mean increase in probe F1 compared to the previous layer.

In [ ]:
# Information gain: probe_f1[:, layer] - probe_f1[:, layer-1]
info_gain = np.zeros(13)
info_gain[0] = probe_f1[:, 0].mean()  # embedding layer vs nothing
for l in range(1, 13):
    gain = (probe_f1[:, l] - probe_f1[:, l - 1])
    info_gain[l] = gain.mean()

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#3498db' if g >= 0 else '#e74c3c' for g in info_gain]
ax.bar(range(13), info_gain, color=colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(13))
ax.set_xticklabels(layer_labels)
ax.set_xlabel('Layer', fontsize=13)
ax.set_ylabel('Mean F1 gain over previous layer', fontsize=13)
ax.set_title('Where BERT Does the Heavy Lifting\n'
             '(mean probe F1 gain per layer, across all emotions)',
             fontsize=14, fontweight='bold')
ax.axhline(y=0, color='black', linewidth=0.5)

# Annotate
for i, g in enumerate(info_gain):
    ax.text(i, g + 0.002 if g >= 0 else g - 0.005, f'{g:+.3f}',
            ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'probing_info_gain.png'), bbox_inches='tight')
plt.show()

print('\nTop 3 layers by information gain:')
top3 = np.argsort(-info_gain)[:3]
for l in top3:
    print(f'  {layer_labels[l]}: {info_gain[l]:+.4f}')

## 7. Connecting Probing to Compression Sensitivity

If an emotion crystallizes at layer L, compressing layer L should be devastating
for that emotion. Let's check this against the lesion study results.

In [ ]:
# Try to load lesion results from NB08
lesion_path = os.path.join(results_dir, 'lesion_per_layer.csv')

if os.path.exists(lesion_path):
    df_lesion = pd.read_csv(lesion_path)
    print(f'Loaded lesion results: {len(df_lesion)} rows')
    
    # Compute mean F1 retention per layer from lesion study
    # and mean probe F1 gain per layer from probing
    layer_probe_importance = probe_f1.mean(axis=0)[1:]  # skip embedding, layers 0-11
    
    # From lesion: compute mean retention per layer
    lesion_retention = []
    for layer_idx in range(12):
        layer_data = df_lesion[df_lesion['layer'] == layer_idx]
        if 'retention' in layer_data.columns:
            mean_ret = layer_data['retention'].mean()
        else:
            mean_ret = layer_data['f1'].mean()
        lesion_retention.append(mean_ret)
    lesion_retention = np.array(lesion_retention)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(layer_probe_importance, lesion_retention,
               s=150, c=range(12), cmap='viridis',
               edgecolors='white', linewidth=2, zorder=3)
    for l in range(12):
        ax.annotate(f'L{l}', (layer_probe_importance[l], lesion_retention[l]),
                    fontsize=11, fontweight='bold', textcoords='offset points', xytext=(8, 5))
    
    rho, p = spearmanr(layer_probe_importance, lesion_retention)
    ax.set_xlabel('Mean probe F1 at layer (probing importance)', fontsize=13)
    ax.set_ylabel('Mean F1 retention under compression (lesion)', fontsize=13)
    ax.set_title(f'Probing vs Compression Sensitivity\n'
                 f'Spearman r = {rho:.3f}, p = {p:.4f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'probing_vs_compression.png'), bbox_inches='tight')
    plt.show()
else:
    print(f'Lesion results not found at {lesion_path}')
    print('Run notebook 08_emotional_map.ipynb first, then re-run this cell.')

## 8. Save Results

In [ ]:
# Save probe F1 matrix
df_probe = pd.DataFrame(probe_f1, index=EMOTION_NAMES, columns=layer_labels)
probe_csv = os.path.join(results_dir, 'probe_results.csv')
df_probe.to_csv(probe_csv)
print(f'Saved: {probe_csv}')

# Save crystallization layers
df_crystal = pd.DataFrame({
    'emotion': EMOTION_NAMES,
    'crystallization_layer': crystal_layer,
    'crystallization_layer_name': [layer_labels[int(cl)] for cl in crystal_layer],
    'max_probe_f1': max_f1_per_emo,
    'train_frequency': emotion_freq,
})
crystal_csv = os.path.join(results_dir, 'crystallization_layers.csv')
df_crystal.to_csv(crystal_csv, index=False)
print(f'Saved: {crystal_csv}')

# Summary
print('\n' + '=' * 65)
print('SUMMARY — Phase 4: Probing Classifiers')
print('=' * 65)
print(f'Probes trained: {probe_f1.shape[0]} emotions x {probe_f1.shape[1]} layers = {probe_f1.size}')
print(f'Overall mean probe F1: {probe_f1[valid_mask].mean():.4f}')
print(f'\nEarliest crystallizers:')
for emo, cl, mf1 in valid_emos[:5]:
    print(f'  {emo:18s}  crystal: {layer_labels[int(cl)]:>4s}  max F1: {mf1:.3f}')
print(f'\nLatest crystallizers:')
for emo, cl, mf1 in valid_emos[-5:]:
    print(f'  {emo:18s}  crystal: {layer_labels[int(cl)]:>4s}  max F1: {mf1:.3f}')
print(f'\nFigures saved to results/:')
for f in ['probing_crystallization_map.png', 'probing_trajectories.png',
          'probing_crystallization_bars.png', 'probing_correlations.png',
          'probing_info_gain.png']:
    status = 'ok' if os.path.exists(os.path.join(results_dir, f)) else 'MISSING'
    print(f'  [{status}] {f}')